## 8. Evaluation: RAGAS, recall@k & Grounding 📊

How to measure RAG quality?
- **RAGAS** → Framework for automated RAG evaluation
- **Recall@k** → Checks if the correct chunk was retrieved
- **Grounding** → Measures how well the response aligns with context


Here's a basic example that breaks down each of the terms you mentioned (RAGAS, Recall\@k, and Grounding) in the context of evaluating RAG (Retrieval-Augmented Generation) models:

---

### 1. **RAGAS** (Retrieval-Augmented Generation Automated Scoring)

### https://docs.ragas.io/en/stable/

**What is RAGAS?**

* RAGAS is a framework designed to automate the evaluation of Retrieval-Augmented Generation (RAG) models.
* It combines multiple factors like relevance, fluency, coherence, and grounding of responses in context.

**Example of RAGAS Evaluation:**
Imagine we are testing a chatbot that retrieves documents and generates a response.

* **Input Query:** "What is the capital of France?"
* **Correct Answer (Ground Truth):** "Paris"

The chatbot retrieves the following document:

> "France is a country in Europe. Its capital city, Paris, is known for its landmarks like the Eiffel Tower."

And the model generates the following response:

> "The capital of France is Paris, located in Europe."

**RAGAS Score Evaluation Criteria:**

* **Relevance**: Does the response answer the query correctly? (Yes)
* **Fluency**: Is the response grammatically sound? (Yes)
* **Coherence**: Does the answer flow logically from the retrieved text? (Yes)
* **Grounding**: Is the response properly grounded in the context provided? (Yes)

RAGAS will give a score based on these factors, possibly with some weight on grounding and relevance.

---

### 2. **Recall\@k**

**What is Recall\@k?**

* Recall\@k measures whether the relevant information or document is retrieved within the top **k** retrieved results.
* In simpler terms, it's checking if the correct chunk of information is among the top k documents or sentences retrieved from a database.

**Example of Recall\@k Evaluation:**

* **Input Query:** "Who developed the theory of relativity?"
* **Correct Answer (Ground Truth):** "Albert Einstein"

The model retrieves the following documents (top 3 retrieved):

1. "Albert Einstein is widely known for developing the theory of relativity."
2. "Isaac Newton developed the laws of motion."
3. "Nikola Tesla contributed to the development of alternating current."

**Recall\@1**: In this case, the correct answer "Albert Einstein" is in the first document. So, Recall\@1 = 1 (i.e., the correct chunk is in the top-1 retrieved chunk).

**Recall\@2**: The correct answer is in the top-1 document as well, so Recall\@2 = 1.

---

### 3. **Grounding**

**What is Grounding?**

* Grounding evaluates how well the generated response is aligned with the information retrieved from the database or corpus.
* It checks if the response is *factually accurate* based on the retrieved context and whether the response has been properly *linked* to the retrieved documents.

**Example of Grounding Evaluation:**

* **Input Query:** "Where is the Eiffel Tower located?"
* **Model’s Generated Response:** "The Eiffel Tower is in Paris, France."

Let's say the model retrieves this chunk of text:

> "The Eiffel Tower is a famous landmark located in Paris."

**Grounding Evaluation:**

* The generated response directly aligns with the information retrieved, so grounding is good in this case.
* If the model had said something like, "The Eiffel Tower is in London," grounding would be poor because it doesn't align with the retrieved document, which states that it's in Paris.

---

### Summary of Example Evaluation

* **RAGAS**: Automated scoring system evaluates the overall quality of the response (relevance, fluency, coherence, grounding).
* **Recall\@k**: Measures if the correct information is within the top **k** retrieved documents. For example, if the top-1 retrieval is correct, Recall\@1 = 1.
* **Grounding**: Evaluates if the generated response is factually aligned with the context or retrieved documents.


In [1]:
# Simulating a simple RAGAS evaluation function

def evaluate_ragas(query, response, retrieved_documents):
    # Simulating document retrieval (usually done by a model)
    # In this case, we're manually defining the documents.
    correct_answer = "Joe Biden is the president of the U.S."

    # Check if the correct information is in the retrieved documents
    relevant_docs = [doc for doc in retrieved_documents if "Joe Biden" in doc]
    
    # RAGAS Evaluation Criteria
    # 1. Relevance: Does the response answer the question correctly?
    relevance = "Joe Biden" in response  # Check if the response mentions the correct answer
    
    # 2. Fluency: Is the response grammatically sound and understandable?
    fluency = isinstance(response, str) and len(response.split()) > 2  # Simple check for fluent response
    
    # 3. Coherence: Does the response match the retrieved documents logically?
    coherence = any(doc for doc in relevant_docs if response in doc)
    
    # 4. Grounding: Is the answer aligned with retrieved documents?
    grounding = any(correct_answer in doc for doc in retrieved_documents)
    
    return {
        "Relevance": relevance,
        "Fluency": fluency,
        "Coherence": coherence,
        "Grounding": grounding
    }

# Example usage:

# Query that the user asks
query = "Who is the president of the United States?"

# Model's generated response
response = "Joe Biden is the current president of the U.S."

# Simulated retrieved documents (from a knowledge base or search engine)
retrieved_documents = [
    "Joe Biden is the president of the U.S.",
    "Barack Obama served as the U.S. president from 2009-2017.",
    "Donald Trump served as the president before Joe Biden."
]

# Evaluate the response using the RAGAS evaluation function
evaluation = evaluate_ragas(query, response, retrieved_documents)

# Display the results of the RAGAS evaluation
print("RAGAS Evaluation Results:")
for metric, score in evaluation.items():
    print(f"{metric}: {'Pass' if score else 'Fail'}")


RAGAS Evaluation Results:
Relevance: Pass
Fluency: Pass
Coherence: Fail
Grounding: Pass


---
---

### Approach Using a Custom RAG :

In [4]:
import openai
import numpy as np
import faiss
from langchain.chat_models import ChatOpenAI
from os import getenv
from dotenv import load_dotenv

load_dotenv(dotenv_path='./data/.env')  # Adjust the path if the .env is outside one folder


# Step 1: Define a simple knowledge base
documents = [
    "The Eiffel Tower is located in Paris, France.",
    "The Great Wall of China is a historic fortification.",
    "The Pyramids of Egypt are ancient structures built by the Egyptians.",
    "Mount Everest is the highest mountain on Earth."
]

# Step 2: Simulate retrieval using a simple text search (we'll use FAISS for vector search)
def get_embedding(text):
    # Simulate embeddings with random vectors (use a real embedding model for production)
    return np.random.rand(512)  # Placeholder for a real embedding

# Create FAISS index for document retrieval
index = faiss.IndexFlatL2(512)  # Using L2 distance for vector search
embeddings = np.array([get_embedding(doc) for doc in documents])
index.add(embeddings)

def retrieve_document(query):
    query_embedding = get_embedding(query)
    D, I = index.search(np.array([query_embedding]), 1)  # Retrieve the closest document
    return documents[I[0][0]]  # Return the most relevant document

# Step 3: Augment generation using OpenAI's API
def generate_answer(query):
    relevant_doc = retrieve_document(query)
    
    # Prepare the prompt with retrieved document
    prompt = f"Query: {query}\nRelevant Info: {relevant_doc}\nAnswer:"
    print(prompt)
    
        
    llm = ChatOpenAI(
    openai_api_base=getenv("OPENROUTER_BASE_URL"),
    openai_api_key=getenv("OPENROUTER_API_KEY"),
    model_name="google/gemma-3-27b-it:free",
    )

    response = llm.invoke(prompt)
    
    # Extract the generated answer
    generated_text = response.content
    return generated_text

# Step 4: Run the system with a query
query = "Where is the Eiffel Tower located?"
generated_answer = generate_answer(query)

print(f"Generated Answer: {generated_answer}")



Query: Where is the Eiffel Tower located?
Relevant Info: The Eiffel Tower is located in Paris, France.
Answer:
Generated Answer: The Eiffel Tower is located in Paris, France.



## Demo project - https://github.com/Kaps-programer-9123/AI_powered_ad_generator/tree/main

## 🏁 Wrap-Up

You learned:
- Document chunking & overlap strategies
- Dense embeddings: E5, BGE, MiniLM
- Vector databases: Chroma, FAISS, Qdrant, Weaviate
- Similarity search (cosine, dot-product)
- Sparse retrieval: BM25, TF-IDF
- Cross-encoder reranking
- Context injection & templating
- Evaluation using RAGAS, recall@k, grounding

## **Next → Topic 4: Fine-Tuning & Adaptation** 🛠️
